# Notebook 01 - Data Preprocessing
## Fake News Detection - NLP Assignment
### Person 2: W.A. Irusha Madushan (CIT-24-01-0514)

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
# Load the WELFake dataset
df = pd.read_csv('../data/WELFake_Dataset.csv')

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Dataset loaded successfully
Shape: (72134, 4)

First 5 rows:


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [3]:
# Basic dataset information
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nClass Distribution:")
print(df['label'].value_counts())
print("\nClass Distribution (%):")
print(df['label'].value_counts(normalize=True) * 100)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  72134 non-null  int64 
 1   title       71576 non-null  object
 2   text        72095 non-null  object
 3   label       72134 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 2.2+ MB
None

Missing Values:
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

Class Distribution:
label
1    37106
0    35028
Name: count, dtype: int64

Class Distribution (%):
label
1    51.440375
0    48.559625
Name: proportion, dtype: float64


## Data Observations
- Dataset contains 72,134 news articles with 4 columns: index, title, text, and label
- Label 1 = Fake News, Label 0 = Real News
- 558 missing titles and 39 missing text values need to be handled
- Classes are well balanced (51% fake, 49% real) — no class imbalance issue
- No missing values in the label column

In [4]:
# Drop rows where text is missing
df = df.dropna(subset=['text'])

# Fill missing titles with empty string
df['title'] = df['title'].fillna('')

# Combine title and text into one column
df['content'] = df['title'] + ' ' + df['text']

print("After handling missing values:")
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum())

After handling missing values:
Shape: (72095, 5)
Missing values:
 Unnamed: 0    0
title         0
text          0
label         0
content       0
dtype: int64


## Text Cleaning
We will now clean the content column by:
- Converting to lowercase
- Removing URLs
- Removing special characters and punctuation
- Removing extra whitespace

In [5]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove special characters, punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning to content column
df['cleaned_content'] = df['content'].apply(clean_text)

print("Sample original text:")
print(df['content'].iloc[0][:200])
print("\nSample cleaned text:")
print(df['cleaned_content'].iloc[0][:200])

Sample original text:
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #F

Sample cleaned text:
law enforcement on high alert following threats against cops and whites on by blacklivesmatter and fyf terrorists video no comment is expected from barack obama members of the fyf or fukyoflag and bla


In [7]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [8]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words]
    return filtered

# Apply tokenization and stopword removal
df['tokens'] = df['cleaned_content'].apply(remove_stopwords)

print("Sample tokens (first 20):")
print(df['tokens'].iloc[0][:20])
print("\nTotal articles after preprocessing:", len(df))

Sample tokens (first 20):
['law', 'enforcement', 'high', 'alert', 'following', 'threats', 'cops', 'whites', 'blacklivesmatter', 'fyf', 'terrorists', 'video', 'comment', 'expected', 'barack', 'obama', 'members', 'fyf', 'fukyoflag', 'blacklivesmatter']

Total articles after preprocessing: 72095


In [9]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

# Apply lemmatization
df['lemmatized'] = df['tokens'].apply(lemmatize_tokens)

# Join tokens back into a string
df['processed_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print("Sample lemmatized tokens (first 20):")
print(df['lemmatized'].iloc[0][:20])
print("\nSample processed text:")
print(df['processed_text'].iloc[0][:200])

Sample lemmatized tokens (first 20):
['law', 'enforcement', 'high', 'alert', 'following', 'threat', 'cop', 'white', 'blacklivesmatter', 'fyf', 'terrorist', 'video', 'comment', 'expected', 'barack', 'obama', 'member', 'fyf', 'fukyoflag', 'blacklivesmatter']

Sample processed text:
law enforcement high alert following threat cop white blacklivesmatter fyf terrorist video comment expected barack obama member fyf fukyoflag blacklivesmatter movement called lynching hanging white pe


## Preprocessing Complete
The following steps were applied to the raw text:
1. Combined title and text into single content column
2. Converted to lowercase
3. Removed URLs, special characters, punctuation and numbers
4. Tokenized text into individual words
5. Removed stopwords (common words like "the", "and", "is")
6. Lemmatized tokens to root form (e.g. "threats" → "threat")

The processed_text column is now ready for feature engineering.

In [10]:
# Keep only needed columns
df_clean = df[['title', 'text', 'content', 'processed_text', 'label']].reset_index(drop=True)

# Save to CSV
df_clean.to_csv('../data/cleaned_data.csv', index=False)

print("Cleaned dataset saved successfully")
print("Shape:", df_clean.shape)
print("\nSample:")
df_clean.head()

Cleaned dataset saved successfully
Shape: (72095, 5)

Sample:


,title,text,content,processed_text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,law enforcement high alert following threat co...,1
1,,Did they post their votes for Hillary already?,Did they post their votes for Hillary already?,post vote hillary already,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,unbelievable obamas attorney general say charl...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,"Bobby Jindal, raised Hindu, uses story of Chri...",bobby jindal raised hindu us story christian c...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",SATAN 2: Russia unvelis an image of its terrif...,satan russia unvelis image terrifying new supe...,1
